# Spark-X2.6-4B-En4B-A1B — 20 h Resumable Long Run (MoE-Engram + mHC, frozen backbone)

Single-epoch-style training over **unique** streamed trajectories, with **leak-free held-out validation**, best-checkpoint selection and plateau rollback.
Designed for **multiple Colab sessions**: all state (curated data chunks, frozen validation set, `latest` / `best` / `final` weights, `state.json`) is persisted to a private Hub repo
(`<HF_REPO_ID>-longrun`). **To resume after a disconnect: re-run Cells 1-3 in the new session.** Optimizer moments are not persisted (LR is re-warmed for a few steps).

* Cell 1 - sync repo + deps  * Cell 2 - load model + config  * Cell 3 - run / resume (one call = one session segment)  * Cell 4 - leak-free post-run diagnostics  * Cell 5 - optional promote


In [ ]:
# ==============================================================================
# Cell 1: Clone / Sync Repository & Install Dependencies
# ==============================================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if Path('/content').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

pip_pkgs = [
    'torch>=2.5.0',
    'triton>=3.1.0',
    'transformers==4.57.1',
    'datasets>=3.2.0',
    'accelerate>=1.2.0',
    'safetensors>=0.4.5',
    'huggingface_hub>=0.27.0',
    'bitsandbytes>=0.45.0',
    'cut-cross-entropy',
    'liger-kernel',
    'unsloth',
    'openai>=1.58.0',
    'nltk',
    'langdetect',
    'psutil',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pip_pkgs], check=False)
print('Working directory synced:', Path.cwd())

In [ ]:
# ==============================================================================
# Cell 2: Load model + long-run configuration
# ==============================================================================
import os, subprocess, importlib, torch
subprocess.run(['git', 'pull', '--ff-only'], check=False)
import sparkx_loop_v03_core, sparkx_loop_v03_train_env, sparkx_engram_generalization, sparkx_long_run
for _m in (sparkx_loop_v03_core, sparkx_loop_v03_train_env, sparkx_engram_generalization, sparkx_long_run):
    importlib.reload(_m)
from load_inference_v03 import load_sparkx_loop_v03
from sparkx_loop_v03_train_env import auto_configure_vram_16k, build_default_cfg

def _get_secret(name):
    v = os.environ.get(name)
    if v:
        return v.strip()
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    return None

HF_TOKEN = _get_secret('HF_TOKEN')                     # needs WRITE access
HF_REPO_ID = 'TassP0lar/Spark-X2.6-4B-En4B-A1B'        # starting weights (the long run never overwrites this repo)
LONGRUN_REPO_ID = HF_REPO_ID + '-longrun'              # private: data chunks, state.json, ckpt/{latest,best,final}

# ---- budget & policy ----------------------------------------------------------
LONGRUN_TOTAL_HOURS    = 20.0   # TRAINING budget over ALL sessions (steps are derived from measured throughput after 12 steps)
LONGRUN_SEGMENT_HOURS  = 22.0   # 24 h Colab cap minus 2 h margin; training stops cleanly (+25 min internal save/push margin) after this long
LONGRUN_EVAL_EVERY     = 40     # steps between held-out evaluations (~1 min each, + anchor audit every 4th)
LONGRUN_PUSH_EVERY     = 80     # steps between Hub pushes of `latest` (+`best`)
LONGRUN_LR_SCALE       = 0.5    # same as Stage4 of the original run
VAL_EVAL_DOCS          = 64     # frozen held-out docs (balanced over the 7 pillars); docs with sha1(prompt)%25==0 are never trained on

model, tokenizer = load_sparkx_loop_v03(
    repo_id=HF_REPO_ID, device='cuda' if torch.cuda.is_available() else 'cpu', token=HF_TOKEN,
    enable_loglinear_kv=True, use_pristine_base=False,
)
CFG = build_default_cfg()
CFG['HF_REPO_ID'] = HF_REPO_ID
CFG['ENABLE_PHASE4_POST_TRAIN'] = False      # these two post-train stages select "best" on texts they train on (leak) -> OFF
CFG['ENABLE_SI_SRSI_ENGRAM_LOOP'] = False
CFG = auto_configure_vram_16k(CFG)
CFG.update(dict(
    LONGRUN_REPO_ID=LONGRUN_REPO_ID, LONGRUN_DIR='/content/longrun',
    LONGRUN_TOTAL_HOURS=LONGRUN_TOTAL_HOURS, LONGRUN_SEGMENT_HOURS=LONGRUN_SEGMENT_HOURS,
    LONGRUN_EVAL_EVERY=LONGRUN_EVAL_EVERY, LONGRUN_PUSH_EVERY=LONGRUN_PUSH_EVERY,
    LONGRUN_LR_SCALE=LONGRUN_LR_SCALE, VAL_EVAL_DOCS=VAL_EVAL_DOCS,
))
print('Ready. HF token present:', bool(HF_TOKEN), '| longrun repo:', LONGRUN_REPO_ID)


In [ ]:
# ==============================================================================
# Cell 3: Run / resume the long run (one call = one session segment). Re-run in a NEW session to continue.
# ==============================================================================
from sparkx_long_run import run_long_training
LONGRUN_SUMMARY = run_long_training(model, tokenizer, CFG, hf_token=HF_TOKEN)
print({k: v for k, v in LONGRUN_SUMMARY.items() if k not in ('state', 'baseline')})


In [ ]:
# ==============================================================================
# Cell 4: Leak-free post-run diagnostics (only meaningful once the run FINISHED or at least produced a best checkpoint)
# ==============================================================================
import json
import sparkx_engram_generalization as eg
st = LONGRUN_SUMMARY['state']
base = st['baseline']
print(f"{'step':>6} {'tokens(M)':>9} {'CE on':>8} {'CE off':>8} {'delta(on-off)':>13} {'vs base(on)':>11}  action")
print(f"{0:>6} {0:>9.1f} {base['ce_on']:>8.4f} {base['ce_off']:>8.4f} {base['engram_delta_ce']:>+13.4f} {0.0:>+11.4f}  baseline")
for r in st['val_history']:
    print(f"{r['step']:>6} {r['tokens']/1e6:>9.1f} {r['ce_on']:>8.4f} {r['ce_off']:>8.4f} {r['engram_delta_ce']:>+13.4f} {r['ce_on']-base['ce_on']:>+11.4f}  {r['action']}")
print('best:', st['best_val_ce_on'], '@ step', st['best_step'])

# Independent 5-text check (handwritten, never in any training set; noise floor ~0.02 nats) on the CURRENT weights
UTIL = eg.engram_utility_report(model, tokenizer)
CV = eg.cross_validated_anchor_gating(model, tokenizer)
print('=== COPY-PASTE LONGRUN JSON ===')
print(json.dumps({'val_history': [{k: r[k] for k in ('step', 'tokens', 'ce_on', 'ce_off', 'engram_delta_ce', 'action')} for r in st['val_history']],
                  'baseline': {k: base[k] for k in ('ce_on', 'ce_off', 'engram_delta_ce')},
                  'best': [st['best_val_ce_on'], st['best_step']],
                  'util5': {k: UTIL[k] for k in ('ce_off', 'ce_full', 'harmful_anchors', 'helpful_anchors')},
                  'cv_gating': CV['cv_ce'], 'stable_anchors': CV['stable_anchors']}, indent=1, default=str))


In [ ]:
# ==============================================================================
# Cell 5 (OPTIONAL): promote the final/best weights to a NEW standalone Hub repo (never overwrites HF_REPO_ID)
# ==============================================================================
PROMOTE = False
PROMOTE_REPO_ID = HF_REPO_ID + '-v2'
if PROMOTE:
    from sparkx_loop_v03_train_env import export_final_v03_to_hub
    CFG['HF_REPO_ID'] = PROMOTE_REPO_ID
    CFG['FINAL_EXPORT_DIR'] = '/content/final_export_longrun'
    CFG['HF_PRIVATE'] = True
    export_final_v03_to_hub(model, tokenizer, CFG, LONGRUN_SUMMARY['global_step'], {'longrun': True}, hf_token=HF_TOKEN)
else:
    print('PROMOTE=False: nothing exported. Weights live in', LONGRUN_REPO_ID, '(ckpt/best, ckpt/final).')
